# SMILES-2026 — Ablation Sweep v2 (Kaggle, 12h GPU)

Расширенная ablation-серия после анализа первой итерации. Главная находка первой серии: AUROC у baseline (last × last × legacy probe) — 0.636, все более сложные фичи (mean-pool, геометрия, топология) AUROC ронят. Только last-token-only улучшил accuracy.

**Эта серия** прицельно проверяет 4 направления:
1. **L-серия (24 прогона):** per-layer scan — каждый из 24 слоёв трансформера по отдельности → найти оптимальную глубину.
2. **M / T / Lin / HP:** комбинации слоёв, tail-pool по ответу, линейный probe (logistic regression), гиперпараметры probe.
3. **K-серия:** 5-fold верификация лучших кандидатов — стабильные AUROC вместо шумного single-split.
4. **Контроли C0/C1:** воспроизведение baseline для калибровки.

**Всего 54 ablation, ~9 часов на T4.**

**Перед запуском (Kaggle Settings справа):**
* **Accelerator:** `GPU T4 x2` или `GPU P100`. Solution.py использует один GPU.
* **Internet:** `On` — нужен для скачивания Qwen2.5-0.5B с HuggingFace при первом запуске.
* **Persistence:** `Files only`.

Все артефакты складываются в `/kaggle/working/ablation_results/`. После каждого прогона обновляется `ablation_results/summary.csv` — если сессия упадёт, частичные результаты не пропадут. Победитель по `test_auroc` копируется в `/kaggle/working/predictions.csv` + `results.json`.

## 1. Проверка окружения

In [ ]:
import sys, os, subprocess
import torch

print(f'Python      : {sys.version.split()[0]}')
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA avail. : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
print()
subprocess.run(['nvidia-smi'], check=False)

## 2. Получение репозитория

Если папка `SMILES-HALLUCINATION-DETECTION` залита как Kaggle Dataset и подключена через **+ Add Input** — она найдётся в `/kaggle/input/<dataset-slug>/`. Иначе клонирую с GitHub (нужен Internet=On).

Поменяй `GIT_REPO`/`GIT_BRANCH` если у тебя fork.

In [ ]:
import shutil
from pathlib import Path

WORK = Path('/kaggle/working/SMILES-HALLUCINATION-DETECTION')
GIT_REPO = 'https://github.com/ahdr3w/SMILES-HALLUCINATION-DETECTION.git'
GIT_BRANCH = None

def find_local_copy() -> Path | None:
    cands = list(Path('/kaggle/input').glob('*/SMILES-HALLUCINATION-DETECTION'))
    cands += list(Path('/kaggle/input').glob('*/*/SMILES-HALLUCINATION-DETECTION'))
    for c in cands:
        if (c / 'solution.py').exists():
            return c
    return None

if WORK.exists():
    print(f'Already prepared at {WORK}')
else:
    src = find_local_copy()
    if src is not None:
        print(f'Copying local copy from {src}')
        shutil.copytree(src, WORK)
    else:
        print(f'Cloning {GIT_REPO} ...')
        cmd = ['git', 'clone', '--depth', '1']
        if GIT_BRANCH:
            cmd += ['--branch', GIT_BRANCH]
        cmd += [GIT_REPO, str(WORK)]
        subprocess.run(cmd, check=True)

os.chdir(WORK)
print(f'cwd: {os.getcwd()}')

### 2.1 Проверка версии файлов

Этот ноутбук рассчитан на репозиторий, в котором уже:
* `aggregation.py` поддерживает `tail_mean`/`tail_max` pool и `AGG_TAIL_N`.
* `probe.py` поддерживает `PROBE_TYPE=linear`.
* `splitting.py` поддерживает `SPLIT_KFOLDS`.
* `solution.py` имеет `USE_GEOMETRIC=True`.

In [ ]:
import re

checks = {
    'solution.USE_GEOMETRIC=True': bool(re.search(r'USE_GEOMETRIC\s*=\s*True', (WORK / 'solution.py').read_text(encoding='utf-8'))),
    'aggregation.AGG_TAIL_N':       'AGG_TAIL_N' in (WORK / 'aggregation.py').read_text(encoding='utf-8'),
    'aggregation.tail_mean':        'tail_mean' in (WORK / 'aggregation.py').read_text(encoding='utf-8'),
    'probe.PROBE_TYPE':             'PROBE_TYPE' in (WORK / 'probe.py').read_text(encoding='utf-8'),
    'splitting.SPLIT_KFOLDS':       'SPLIT_KFOLDS' in (WORK / 'splitting.py').read_text(encoding='utf-8'),
}
for name, ok in checks.items():
    mark = 'OK' if ok else 'FAIL'
    print(f'  [{mark}] {name}')
if not all(checks.values()):
    raise RuntimeError('Репозиторий не обновлён. Подложи свежие версии файлов и перезапусти.')

## 3. Установка зависимостей

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=False)

## 4. Описание ablation-серии

**Семь групп, 54 прогона.** Все используют `GEOM_*=0` (геометрия отключена — доказано вредна в первой итерации). MLP-probe по умолчанию (2 hidden GELU + dropout 0.3 + early stopping). Single split, кроме K-серии (K=5).

### C — контроли (2)
| Tag | LAYERS | POOLS | Probe | feature_dim |
|---|---|---|---|---|
| `C0_baseline` | -1 | last | legacy | 896 |
| `C1_last1_mlp` | -1 | last | mlp | 896 |

### L — per-layer scan, last-token only, MLP (24)
По одному слою от -24 до -1. Цель — кривая `AUROC ↔ глубина`.

### M — multi-layer комбинации, last-token (6)
`M1_last2 (-2,-1)`, `M2_last4 (-4..-1)`, `M3_last8`, `M4_last16`, `M5_skip3 (-12,-8,-4,-1)`, `M6_wide (-20,-16,-12,-8,-4,-1)`.

### T — tail-pool по ответу (8)
Mean/max по последним N реальным токенам (эвристика «pool по response» без токенайзера). N ∈ {8, 16, 32, 64}, layer = -1 кроме T7 (last4) и T8 (mid).

### Lin — linear probe (4)
Один `nn.Linear(in, 1)` (= logistic regression) на разных подмножествах слоёв.

### HP — probe hyperparameter tune (4)
Базовый конфиг M2 (last4), варьируем по одному гиперпараметру (hidden 128/512, dropout 0.5, layers 3).

### K — 5-fold верификация (6)
Разнообразные кандидаты с `SPLIT_KFOLDS=5` для устойчивого ранжирования.

In [ ]:
# Базовый набор «выключателей геометрии», который добавляется ко всем new-style ablation.
NOGEO = {
    'GEOM_NORMS': '0', 'GEOM_DRIFT': '0', 'GEOM_TOKEN_STATS': '0',
    'GEOM_LAST_VS_MEAN': '0', 'GEOM_SEQ_LEN': '0', 'GEOM_GLOBAL_DRIFT': '0',
    'GEOM_TOPOLOGY': '0',
}

ABLATIONS: list[tuple[str, dict]] = []

# ============== C: контроли ==============
ABLATIONS.append(('C0_baseline', {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'last', 'PROBE_LEGACY': '1'}))
ABLATIONS.append(('C1_last1_mlp', {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'last'}))

# ============== L: per-layer scan (24 слоя) ==============
# L_l01 -> LAYERS=-24 (самый ранний transformer-слой, после embedding)
# L_l24 -> LAYERS=-1  (последний)
for i, layer in enumerate(range(-24, 0), start=1):
    ABLATIONS.append((f'L_l{i:02d}', {**NOGEO, 'AGG_LAYERS': str(layer), 'AGG_POOLS': 'last'}))

# ============== M: multi-layer комбинации ==============
ABLATIONS.append(('M1_last2',  {**NOGEO, 'AGG_LAYERS': '-2,-1', 'AGG_POOLS': 'last'}))
ABLATIONS.append(('M2_last4',  {**NOGEO, 'AGG_LAYERS': '-4,-3,-2,-1', 'AGG_POOLS': 'last'}))
ABLATIONS.append(('M3_last8',  {**NOGEO, 'AGG_LAYERS': '-8,-7,-6,-5,-4,-3,-2,-1', 'AGG_POOLS': 'last'}))
ABLATIONS.append(('M4_last16', {**NOGEO, 'AGG_LAYERS': ','.join(str(-i) for i in range(16, 0, -1)), 'AGG_POOLS': 'last'}))
ABLATIONS.append(('M5_skip3',  {**NOGEO, 'AGG_LAYERS': '-12,-8,-4,-1', 'AGG_POOLS': 'last'}))
ABLATIONS.append(('M6_wide',   {**NOGEO, 'AGG_LAYERS': '-20,-16,-12,-8,-4,-1', 'AGG_POOLS': 'last'}))

# ============== T: tail-pool по ответу ==============
ABLATIONS.append(('T1_tail8',       {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'tail_mean', 'AGG_TAIL_N': '8'}))
ABLATIONS.append(('T2_tail16',      {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'tail_mean', 'AGG_TAIL_N': '16'}))
ABLATIONS.append(('T3_tail32',      {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'tail_mean', 'AGG_TAIL_N': '32'}))
ABLATIONS.append(('T4_tail64',      {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'tail_mean', 'AGG_TAIL_N': '64'}))
ABLATIONS.append(('T5_tail_max16',  {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'tail_max', 'AGG_TAIL_N': '16'}))
ABLATIONS.append(('T6_combo_last1', {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'last,tail_mean', 'AGG_TAIL_N': '16'}))
ABLATIONS.append(('T7_combo_last4', {**NOGEO, 'AGG_LAYERS': '-4,-3,-2,-1', 'AGG_POOLS': 'last,tail_mean', 'AGG_TAIL_N': '16'}))
ABLATIONS.append(('T8_tail_mid',    {**NOGEO, 'AGG_LAYERS': '-12', 'AGG_POOLS': 'tail_mean', 'AGG_TAIL_N': '16'}))

# ============== Lin: linear probe ==============
ABLATIONS.append(('Lin1_last1', {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'last', 'PROBE_TYPE': 'linear'}))
ABLATIONS.append(('Lin2_last4', {**NOGEO, 'AGG_LAYERS': '-4,-3,-2,-1', 'AGG_POOLS': 'last', 'PROBE_TYPE': 'linear'}))
ABLATIONS.append(('Lin3_mid',   {**NOGEO, 'AGG_LAYERS': '-12', 'AGG_POOLS': 'last', 'PROBE_TYPE': 'linear'}))
ABLATIONS.append(('Lin4_wide',  {**NOGEO, 'AGG_LAYERS': '-20,-16,-12,-8,-4,-1', 'AGG_POOLS': 'last', 'PROBE_TYPE': 'linear'}))

# ============== HP: probe hyperparameter tune (база = M2) ==============
_HP_BASE = {**NOGEO, 'AGG_LAYERS': '-4,-3,-2,-1', 'AGG_POOLS': 'last'}
ABLATIONS.append(('HP1_hidden128', {**_HP_BASE, 'PROBE_HIDDEN': '128'}))
ABLATIONS.append(('HP2_hidden512', {**_HP_BASE, 'PROBE_HIDDEN': '512'}))
ABLATIONS.append(('HP3_dropout05', {**_HP_BASE, 'PROBE_DROPOUT': '0.5'}))
ABLATIONS.append(('HP4_layers3',   {**_HP_BASE, 'PROBE_LAYERS': '3'}))

# ============== K: 5-fold верификация ==============
_KF = {'SPLIT_KFOLDS': '5'}
ABLATIONS.append(('K1_C0_5fold',   {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'last', 'PROBE_LEGACY': '1', **_KF}))
ABLATIONS.append(('K2_L24_5fold',  {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'last', **_KF}))
ABLATIONS.append(('K3_L13_5fold',  {**NOGEO, 'AGG_LAYERS': '-12', 'AGG_POOLS': 'last', **_KF}))
ABLATIONS.append(('K4_M2_5fold',   {**NOGEO, 'AGG_LAYERS': '-4,-3,-2,-1', 'AGG_POOLS': 'last', **_KF}))
ABLATIONS.append(('K5_T6_5fold',   {**NOGEO, 'AGG_LAYERS': '-1', 'AGG_POOLS': 'last,tail_mean', 'AGG_TAIL_N': '16', **_KF}))
ABLATIONS.append(('K6_Lin2_5fold', {**NOGEO, 'AGG_LAYERS': '-4,-3,-2,-1', 'AGG_POOLS': 'last', 'PROBE_TYPE': 'linear', **_KF}))

# Подмножество тегов для прогона. Пусто = все 54.
TAG_FILTER: set[str] = set()

n_planned = len([t for t, _ in ABLATIONS if not TAG_FILTER or t in TAG_FILTER])
print(f'Запланировано прогонов: {n_planned} / {len(ABLATIONS)}')
print(f'Пример первых 5 тегов: {[t for t,_ in ABLATIONS[:5]]}')
print(f'Пример последних 5 тегов: {[t for t,_ in ABLATIONS[-5:]]}')

## 5. Запуск ablation-серии

Каждый прогон — `python solution.py` в подпроцессе с уникальными ENV. Чекпойнтинг: если для тега `tag` уже есть `ablation_results/results_{tag}.json`, прогон пропускается (полезно при reconnect Kaggle-сессии). После каждого прогона `ablation_results/summary.csv` обновляется.

In [ ]:
import time
import json
import pandas as pd

OUT = WORK / 'ablation_results'
OUT.mkdir(exist_ok=True)

def update_summary() -> pd.DataFrame:
    rows = []
    for f in sorted(OUT.glob('results_*.json')):
        try:
            j = json.load(open(f, 'r', encoding='utf-8'))
        except Exception:
            continue
        rows.append({
            'tag':         f.stem.replace('results_', ''),
            'test_acc':    j.get('avg_test_accuracy'),
            'test_auroc':  j.get('avg_test_auroc'),
            'test_f1':     j.get('avg_test_f1'),
            'val_auroc':   j.get('avg_val_auroc'),
            'train_auroc': j.get('avg_train_auroc'),
            'feature_dim': j.get('feature_dim'),
            'n_folds':     j.get('n_folds'),
            'baseline_acc': j.get('avg_baseline_accuracy'),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['test_auroc', 'test_acc'], ascending=False)
        df.to_csv(OUT / 'summary.csv', index=False)
    return df

def run_one(tag: str, env_overrides: dict[str, str]) -> dict:
    if TAG_FILTER and tag not in TAG_FILTER:
        return {'tag': tag, 'status': 'skipped'}

    res_file = OUT / f'results_{tag}.json'
    if res_file.exists() and res_file.stat().st_size > 0:
        print(f'\n[{tag}] уже есть результат — пропускаю (checkpoint)')
        return {'tag': tag, 'status': 'cached'}

    full_env = os.environ.copy()
    full_env.update(env_overrides)
    full_env.setdefault('TQDM_DISABLE', '1')
    full_env.setdefault('TRANSFORMERS_VERBOSITY', 'error')
    full_env.setdefault('TOKENIZERS_PARALLELISM', 'false')
    # Без HF_HUB_DOWNLOAD_TIMEOUT первый запуск может зависать на медленной сети.
    full_env.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '120')

    print(f'\n========== {tag} ==========')
    print(f'env: {env_overrides}')

    log_path = OUT / f'log_{tag}.txt'
    t0 = time.time()
    with open(log_path, 'w', encoding='utf-8') as logf:
        proc = subprocess.Popen(
            [sys.executable, 'solution.py'],
            cwd=str(WORK),
            env=full_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line); sys.stdout.flush()
            logf.write(line)
        ret = proc.wait()
    elapsed = time.time() - t0

    info = {'tag': tag, 'elapsed_s': elapsed, 'returncode': ret}
    if ret == 0:
        shutil.copy(WORK / 'results.json',    res_file)
        shutil.copy(WORK / 'predictions.csv', OUT / f'predictions_{tag}.csv')
        info['status'] = 'ok'
        print(f'<<< {tag}  OK  ({elapsed:.0f}s)')
    else:
        info['status'] = 'fail'
        print(f'<<< {tag}  FAIL  ({elapsed:.0f}s) — лог: {log_path}')
    update_summary()
    return info

summary_log: list[dict] = []
global_t0 = time.time()
for tag, env_overrides in ABLATIONS:
    summary_log.append(run_one(tag, env_overrides))
global_elapsed = time.time() - global_t0

print(f'\n=== ВСЁ ===  суммарно: {global_elapsed/60:.1f} мин ({global_elapsed/3600:.2f} ч)')
for s in summary_log:
    elapsed = s.get('elapsed_s', 0)
    print(f'  {s["tag"]:25s}  {s.get("status", "?"):8s}  {elapsed:6.0f}s')

## 6. Сводная таблица результатов

Сортировка по `test_auroc`, потому что `test_acc` у многих конфигураций залипает на baseline (artifact F1-threshold tuning). AUROC показывает реальный сигнал.

In [ ]:
df = update_summary()
if df.empty:
    raise RuntimeError('Нет успешных прогонов — проверь ablation_results/log_*.txt')
df.head(30)

In [ ]:
# Подвыборка по группам — отдельные таблицы, чтобы быстро посмотреть кривую и т.д.
groups = {
    'L (per-layer scan)': df[df['tag'].str.startswith('L_l')].sort_values('tag'),
    'M (multi-layer)':    df[df['tag'].str.startswith('M')],
    'T (tail-pool)':      df[df['tag'].str.startswith('T')],
    'Lin (linear)':       df[df['tag'].str.startswith('Lin')],
    'HP (probe tune)':    df[df['tag'].str.startswith('HP')],
    'K (5-fold)':         df[df['tag'].str.startswith('K')],
    'C (control)':        df[df['tag'].str.startswith('C')],
}
for name, g in groups.items():
    if g.empty:
        continue
    print(f'\n=== {name} ({len(g)} прогонов) ===')
    print(g[['tag','test_acc','test_auroc','test_f1','train_auroc','feature_dim','n_folds']].to_string(index=False))

In [ ]:
# Опциональный график AUROC по глубине слоя для L-серии.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

L = df[df['tag'].str.startswith('L_l')].copy()
if not L.empty:
    L['layer_idx'] = L['tag'].str.extract(r'L_l(\d+)').astype(int)
    # L_l01 = -24 (самый ранний transformer-слой), L_l24 = -1.
    L['transformer_layer'] = L['layer_idx']  # 1..24, считая от первого transformer-слоя
    L = L.sort_values('transformer_layer')
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(L['transformer_layer'], L['test_auroc'], '-o', label='test AUROC')
    ax.set_xlabel('transformer layer index (1 = earliest, 24 = last)')
    ax.set_ylabel('test AUROC (single split)')
    ax.set_title('Per-layer scan — last-token only')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(1, 25))
    fig.tight_layout()
    fig.savefig(OUT / 'per_layer_auroc.png', dpi=120)
    plt.show()
    print(f'Сохранён график: {OUT / "per_layer_auroc.png"}')
else:
    print('L-серия пуста — график не строю.')

## 7. Выбор победителя и финальный сабмит

Стратегия отбора:
1. **Если K-серия завершилась** — выбираем лучшего из K-серии (стабильный 5-fold AUROC).
2. **Иначе** — лучшего по single-split test_auroc.
3. Тай-брейк — test_acc.

`predictions.csv` копируется из `ablation_results/predictions_<best>.csv` в `/kaggle/working/predictions.csv`.

In [ ]:
K_df = df[df['tag'].str.startswith('K')]
if not K_df.empty:
    pool = K_df
    print(f'Выбираю из K-серии ({len(K_df)} конфигов с 5-fold)')
else:
    pool = df
    print('K-серия пуста — выбираю из всех single-split')

best_row = pool.sort_values(['test_auroc', 'test_acc'], ascending=False).iloc[0]
best_tag = best_row['tag']
print(f'\nЛучшая конфигурация: {best_tag}')
print(f'  test_auroc  = {best_row["test_auroc"]:.4f}')
print(f'  test_acc    = {best_row["test_acc"]:.4f}')
print(f'  test_f1     = {best_row["test_f1"]:.4f}')
print(f'  train_auroc = {best_row["train_auroc"]:.4f}  (gap = {best_row["train_auroc"] - best_row["test_auroc"]:.3f})')
print(f'  feature_dim = {best_row["feature_dim"]}')
print(f'  n_folds     = {best_row["n_folds"]}')

shutil.copy(OUT / f'predictions_{best_tag}.csv', '/kaggle/working/predictions.csv')
shutil.copy(OUT / f'results_{best_tag}.json',    '/kaggle/working/results.json')
shutil.copy(OUT / 'summary.csv',                 '/kaggle/working/ablation_summary.csv')

# Дополнительно: лучший по test_acc отдельно (вдруг отличается).
best_acc_tag = df.sort_values(['test_acc', 'test_auroc'], ascending=False).iloc[0]['tag']
if best_acc_tag != best_tag:
    shutil.copy(OUT / f'predictions_{best_acc_tag}.csv', f'/kaggle/working/predictions_best_acc_{best_acc_tag}.csv')
    print(f'\nЛучший по test_acc отличается ({best_acc_tag}) — скопировал отдельно.')

print('\n/kaggle/working/:')
for p in sorted(Path('/kaggle/working').iterdir()):
    if p.is_file():
        print(f'  {p.name}  ({p.stat().st_size:,} bytes)')

## 8. Что дальше

1. Скачай `predictions.csv`, `results.json`, `ablation_summary.csv` (а лучше всю `ablation_results/`) с панели Output.
2. `predictions.csv` — финальный сабмит. `ablation_summary.csv` — таблица всех 54 прогонов для SOLUTION.md.
3. График `per_layer_auroc.png` отвечает «где сидит сигнал по глубине» — это самый интересный артефакт для отчёта.
4. Если K-серия показала, что 5-fold AUROC заметно отличается от single-split — single-split цифры были шумом, и доверять стоит K-серии.
5. Если сессия упала — просто перезапусти ноутбук, чекпойнтинг подхватит уже выполненные прогоны.